# 5.2 Human-in-the-Loop Approvals

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

**Prerequisites:** 4.4 (`interrupt()`, `__interrupt__`, `Command(resume=...)`), 5.1 (checkpointer types, `thread_id`). No API key needed: the model in every demo is a scripted stand-in, so the runs are deterministic and free.

## Learning objectives

- Configure `HumanInTheLoopMiddleware` with `interrupt_on`: per-tool policies using `allowed_decisions`, `description`, and `when` predicates.
- Read the interrupt payload (`action_requests`, `review_configs`) and turn it into a reviewer UI contract.
- Execute the three decisions, approve, edit, reject, with `Command(resume=...)`, and explain exactly what the model sees after each one.
- Contrast the static `interrupt_before=["tools"]` policy with per-tool `interrupt_on`, including how each pause is detected and resumed.
- Design the surrounding system: the approval payload, the audit trail, timeouts and escalation, and where approval gates belong.

## The three decisions as a pattern

Every approval gate in an agent system reduces to the same three verdicts. A human looks at a proposed tool call and either accepts it as-is, changes it, or refuses it:

| Decision | Meaning | The tool | The model then sees |
|---|---|---|---|
| `approve` | run it exactly as proposed | executes with the model's arguments | the tool's normal result |
| `edit` | run it, but with my corrections | executes with the reviewer's arguments | the result, plus a note that a human changed the call |
| `reject` | do not run it | never executes | an error message carrying the reviewer's reason |

A fourth decision, `respond`, exists for a different job: tools whose real implementation is a human's answer (an `ask_user` tool). `respond` skips the tool and returns the human's message as a successful result. Never use `respond` to refuse a side-effecting action: the model reads it as success, which is the opposite of a refusal.

Lesson 4.4 taught the raw primitive, `interrupt()` inside a node. That works, but the policy lives in your node code: which tools pause, what the reviewer may decide, what the reviewer sees. `HumanInTheLoopMiddleware` moves the policy into configuration, per tool, which is where it belongs once more than one tool needs different rules.

## The middleware and `interrupt_on`

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    tools=[issue_refund, lookup_order],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "issue_refund": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "description": "Refunds move real money and cannot be undone.",
                },
                "lookup_order": False,
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

`interrupt_on` maps each tool name to one of three things:

- `True`: interrupt with the default config (all four decisions allowed).
- `False`: never interrupt; the tool runs without review.
- An `InterruptOnConfig` dict with:
  - `allowed_decisions`: the list of verdicts the reviewer may give: `"approve"`, `"edit"`, `"reject"`, `"respond"`.
  - `description`: a string (or a callable) shown to the reviewer, explaining what the tool does and why it needs a human. This is the most underused field: a good description is the difference between a reviewer who decides well and one who rubber-stamps.
  - `when`: an optional predicate over the `ToolCallRequest` returning `True` to interrupt, `False` to auto-approve. This is how a gate becomes data-dependent: only refunds above 50 EUR reach a human.
  - `args_schema`: rarely needed; constrains how the reviewer may edit.

Two constructor options worth knowing: `description_prefix` (default `"Tool execution requires approval"`) heads every review request, and `interrupt_mode`, which is `"batched"` by default and `"per_call"` for one interrupt per tool call (needs `langchain>=1.4.6`; see the per-call section below). The middleware needs a checkpointer, for the same reason raw `interrupt()` does: the pause is a saved state that a later call resumes.

## What the human sees: the interrupt payload

When the model proposes a gated tool call, the run pauses and `invoke` returns the state with an `__interrupt__` key. Its value is the review request, and it has a fixed shape your frontend can rely on:

In [ ]:
paused = agent.invoke({"messages": [HumanMessage("Please refund 120 EUR to customer C-7781.")]}, config)
payload = paused["__interrupt__"][0].value
print(payload["action_requests"])
# [{'name': 'issue_refund',
#   'args': {'customer_id': 'C-7781', 'amount_eur': 120.0},
#   'description': 'Refunds move real money and cannot be undone.'}]
print(payload["review_configs"])
# [{'action_name': 'issue_refund', 'allowed_decisions': ['approve', 'edit', 'reject']}]

`action_requests` is what the agent wants to do: tool name, exact arguments, your description. `review_configs` is what the reviewer may do about each request, in the same order. Design your approval card from these two lists and nothing else: the tool name and args are the decision's inputs, the description is the context, the allowed decisions are the buttons. Do not dump the whole state into the card; the reviewer needs the proposal, not the conversation.

## The three decisions, runnable

The demo below uses a scripted model instead of an LLM, so you can run it now and get the same result every time. The scripted model emits one tool call, then a final message. (`GenericFakeChatModel` does not implement `bind_tools`, which `create_agent` requires, so the stand-in subclasses it with a no-op. In production this is a real chat model; nothing else changes.)

In [ ]:
import itertools
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

LEDGER = []  # every real side effect lands here: the audit trail

class ScriptedModel(GenericFakeChatModel):
    """Stand-in for an LLM: bind_tools is a no-op, tool calls are scripted."""

    def bind_tools(self, tools, **kwargs):
        return self

def issue_refund(customer_id: str, amount_eur: float) -> str:
    """Issue a refund to the customer's original payment method."""
    LEDGER.append({"customer_id": customer_id, "amount_eur": amount_eur})
    return f"Refund of {amount_eur:.2f} EUR issued to {customer_id}."

def lookup_order(order_id: str) -> str:
    """Look up an order. Read-only."""
    return f"Order {order_id}: delivered."

def build_agent():
    scripted = [
        AIMessage(content="", tool_calls=[{"name": "issue_refund",
            "args": {"customer_id": "C-7781", "amount_eur": 120.0},
            "id": "call_1", "type": "tool_call"}]),
        AIMessage(content="Done, I have handled your request."),
    ]
    model = ScriptedModel(messages=itertools.cycle(iter(scripted)))
    return create_agent(
        model,
        tools=[issue_refund, lookup_order],
        middleware=[HumanInTheLoopMiddleware(interrupt_on={
            "issue_refund": {
                "allowed_decisions": ["approve", "edit", "reject"],
                "description": "Refunds move real money and cannot be undone.",
            },
            "lookup_order": False,
        })],
        checkpointer=InMemorySaver(),
    )

Each scenario runs on its own thread, pauses on the refund, and resumes with one decision:

In [ ]:
def run_case(thread_id, decision):
    agent = build_agent()
    LEDGER.clear()
    config = {"configurable": {"thread_id": thread_id}}
    paused = agent.invoke(
        {"messages": [HumanMessage("Please refund 120 EUR to customer C-7781.")]},
        config,
    )
    assert "__interrupt__" in paused
    done = agent.invoke(Command(resume={"decisions": [decision]}), config)
    print(thread_id, "-> ledger:", LEDGER)
    return done

run_case("t-approve", {"type": "approve"})
run_case("t-edit", {"type": "edit", "edited_action": {
    "name": "issue_refund", "args": {"customer_id": "C-7781", "amount_eur": 60.0}}})
run_case("t-reject", {"type": "reject", "message": "Outside the 30-day refund window."})

In [ ]:
t-approve -> ledger: [{'customer_id': 'C-7781', 'amount_eur': 120.0}]
t-edit    -> ledger: [{'customer_id': 'C-7781', 'amount_eur': 60.0}]
t-reject  -> ledger: []

The ledger is the ground truth: approve ran the call as proposed, edit ran it with the reviewer's arguments, reject ran nothing. Now the part most tutorials skip: what the model sees next, because that determines what the agent does after the verdict. Reading the tool messages from each run:

- **Approve:** the model gets the normal tool result: `'Refund of 120.00 EUR issued to C-7781.'` with status `success`. Nothing special; the call simply happened.
- **Edit:** the model gets the result prefixed with a notice: `"Note: a human reviewer replaced this tool call before it ran... Executed instead: issue_refund with arguments {...60.0...}"`, status `success`. The notice matters: without it the model would believe its original call ran and might act on the wrong amount. You can change the notice with the middleware's `edit_notice` parameter, or pass `None` to remove it; the default is the safe choice.
- **Reject:** the model gets `'User rejected the tool call for `issue_refund` with reason: Outside the 30-day refund window.'` with status `error`. The tool never ran. Your `message` is the whole explanation the agent gets, so write it as an instruction, not a label: "outside the window, offer store credit instead" beats "rejected".

In batched mode (the default), one interrupt covers every gated call the model made in that turn, and `decisions` lists one verdict per call in the same order as `action_requests`. If the model proposes two refunds in one turn, you send two decisions.

## Data-dependent gating with `when`

Gating every call of a tool is usually wrong: it trains reviewers to click approve without reading. The `when` predicate gates on the call's arguments, so only the calls that need a human ever pause:

In [ ]:
from langchain.agents.middleware import ToolCallRequest

def over_fifty(request: ToolCallRequest) -> bool:
    return request.tool_call["args"].get("amount_eur", 0) > 50

HumanInTheLoopMiddleware(interrupt_on={
    "issue_refund": {"allowed_decisions": ["approve", "reject"], "when": over_fifty},
})

A 20 EUR refund now runs without interrupting; a 120 EUR refund pauses. Two rules keep this honest. First, the predicate runs again on resume, so it must return the same answer for the same call: a predicate that reads the clock or a mutable flag can flip between pause and resume, and if it returns `False` on resume the tool runs and the reviewer's answer is silently ignored, even a reject. Keep predicates pure functions of the tool call. Second, gate on the arguments, not on vibes: "amount over 50" is reviewable policy; a predicate that shells out to another model is a second agent hiding inside your gate.

## Static `interrupt_before=["tools"]` versus per-tool `interrupt_on`

`create_agent` also accepts the static breakpoints from lesson 4.4 directly:

In [ ]:
agent = create_agent(model, tools=[issue_refund], checkpointer=InMemorySaver(),
                     interrupt_before=["tools"])

Every run now pauses before the tools node, for every tool, unconditionally. The pause looks different from a middleware interrupt, and the difference bites if you mix them up:

| | `interrupt_before=["tools"]` | `HumanInTheLoopMiddleware` |
|---|---|---|
| What pauses | every tool call, unconditionally | only the tools in `interrupt_on`, subject to `when` |
| Reviewer sees | no `__interrupt__` key; the raw state | an `__interrupt__` payload with `action_requests` and `review_configs` |
| How you detect the pause | `graph.get_state(config).next == ("tools",)` | `"__interrupt__" in result` |
| How you resume | plain `agent.invoke(None, config)` | `agent.invoke(Command(resume={"decisions": [...]}), config)` |
| Edit/reject | not supported; it is a breakpoint, not a verdict | approve, edit, reject, respond per call |

The static breakpoint is a debugging and blanket-policy tool: "pause before any tool while I develop" or "nothing executes until the operator releases it". The middleware is the approval workflow: per-tool rules, reviewer-shaped payloads, and verdicts the agent understands. A common production shape is both: the middleware for the real policy, and `interrupt_before` kept for staging environments.

## Per-call mode

The default batched mode raises one interrupt per model turn. `interrupt_mode="per_call"` raises one interrupt per gated tool call, as the call starts. Each interrupt's value carries `type: "tool_approval"` plus the `tool_call_id`, `name`, and `args`, so a client can tell approval interrupts apart from other interrupts. You answer each one keyed by its interrupt id:

In [ ]:
HumanInTheLoopMiddleware(interrupt_on={"issue_refund": True}, interrupt_mode="per_call")
# ...
paused = agent.invoke({"messages": [...]}, config)
interrupt_id = paused["__interrupt__"][0].id
agent.invoke(Command(resume={interrupt_id: {"type": "approve"}}), config)

Per-call mode needs `langchain>=1.4.6`, and the docs recommend it for new agents: batched mode's ordered-decision list is fragile when calls are added or removed between pause and resume, while an id-keyed answer stays attached to its call. An interrupt you do not answer stays pending, and its tool runs once the answer arrives, so a reviewer can work through a queue of calls in any order. The middleware ordering rule matters here: list `HumanInTheLoopMiddleware` before retry and error-handling middleware, otherwise a retried call asks the reviewer again.

## Designing the approval payload

The payload is a user interface, and most of them are bad: they show the reviewer a JSON blob and three identical buttons. Four rules from systems that do this well:

1. **Show the decision's inputs, not the conversation.** Tool name, arguments, your `description`. The reviewer decides on the action, not on the chat history.
2. **Say what happens on each verdict.** "Approve runs the refund now" next to the button. Reviewers should never have to guess whether approve means "run it" or "looks fine, keep thinking".
3. **Make edit safe by construction.** `edited_action` must keep the tool name and match the tool's argument schema; unknown arguments are ignored. Your UI should offer the editable fields (amount, recipient), not a free-text JSON box, because a typo in a JSON box becomes a failed or wrong tool call.
4. **Record the verdict where it cannot be lost.** The message history already carries the outcome (the tool result or the rejection), but a compliance trail wants more: who decided, when, and what they saw. Append to your own log at decision time: thread id, tool name, arguments as proposed, verdict, edited arguments if any, reviewer identity, timestamp. The framework does not do this for you.

## Timeouts and escalation: the human who never answers

Nothing in the framework expires a paused thread. A run that waits for a reviewer on Friday evening is still waiting on Monday morning, holding no CPU or RAM (the state is checkpointed; the process is free), but the user is still waiting too. Design for it:

- **Put an expiry in the payload, not in the framework.** Include `expires_at` in what you show the reviewer and in your own records.
- **Run a sweeper.** A scheduled job lists threads paused longer than the SLA and resumes them with a safe default, usually `reject` with a message like "no reviewer available within 15 minutes; the action was cancelled". Cancelling is the safe default; auto-approving on timeout is how incidents happen.
- **Escalate, do not just wait.** Route unanswered approvals up: notify a second reviewer, page the on-call, or degrade to a narrower action the agent may take alone.
- **Tell the user the truth.** "Waiting for a human reviewer" with the queue position beats a spinner. If the sweeper cancels, say so and say what happens next.

## Where approval gates belong

Gate irreversible side effects: moving money, deleting data, sending messages to customers, changing production systems. Do not gate reads: a reviewer clicking approve on every search query learns to click approve on everything, and your refund gate inherits that habit. The `lookup_order: False` in the demo is the point, not an aside. When everything needs approval, approval means nothing; when only the dangerous three percent pauses, reviewers actually read.

## The EU angle: oversight you can evidence

For high-risk deployments, the EU AI Act expects human oversight to be designed in, not bolted on. An approval gate is one concrete implementation of that: a person reviews the specific action before it executes. But the gate alone is not the compliance story; the audit trail is. "A human approved it" is a claim; the ledger entry with the thread id, the proposed arguments, the verdict, the reviewer identity, and the timestamp is evidence. Keep that trail under the same retention and access rules as the checkpoints from lesson 5.1: it contains personal data (who did what to whose account), it needs a deletion policy, and it needs access control. Oversight without records is theatre.

## What can go wrong

- **No checkpointer, no resume.** Same rule as raw `interrupt()`: the pause looks fine, the resume fails. The middleware does not add one for you; pass `checkpointer` to `create_agent`.
- **Gating reads alongside writes.** Reviewers fatigue, approvals become reflexes, and the gate on the dangerous tool stops meaning anything. Gate irreversible actions only.
- **Using `respond` to refuse.** `respond` returns the human's message as a successful tool result. Refusing a side-effecting call with `respond` tells the model the action succeeded. Refusals are `reject`, always.
- **Impure `when` predicates.** The predicate re-runs on resume. If it flips to `False`, the tool executes and the reviewer's verdict is discarded silently. Predicates must be pure functions of the tool call.
- **Editing with a free-text JSON box.** `edited_action` keeps the tool name and must match the argument schema; a malformed edit fails or, worse, runs with ignored fields. Offer structured fields in the UI.
- **Answering batched decisions in the wrong order.** In batched mode, `decisions` aligns positionally with `action_requests`. A shifted list approves the wrong call. Per-call mode removes this failure entirely.
- **Detecting a static breakpoint like a middleware interrupt.** `interrupt_before` produces no `__interrupt__` key; checking for the key tells you the run finished. Read `get_state(config).next` for static pauses.
- **The thread that waits forever.** No built-in timeout exists. Without a sweeper and an escalation path, approvals die quietly in the queue and users wait indefinitely. Cancel-safe is the default; auto-approve on timeout is an incident waiting for a date.
- **No audit trail beyond the chat.** The message history shows what happened but not who decided or what they saw. For anything regulated, log the verdict, the reviewer, and the proposed arguments in your own store at decision time.
- **Approving without reading the args.** The classic failure is social, not technical: the reviewer approves the tool name instead of the arguments. The payload design section exists because the arguments are the decision.

## Check your understanding

**1. The reviewer clicks "approve" on a refund the model proposed. The tool runs. What does the model see next, and why does the exact content matter?**
The model sees the tool's normal result with status `success`, exactly as if no human had been involved. The content matters because the agent's next step is built on it: with `edit`, the notice that a human replaced the call stops the model from acting on its original arguments; with `reject`, the reason you write becomes the agent's instruction for what to do instead.

**2. You configured `interrupt_on={"send_email": True}` and the run pauses, but the payload's `review_configs` allows all four decisions. You only want approve and reject. What do you change, and why not just hide the extra buttons in the UI?**
Change the config to `{"send_email": {"allowed_decisions": ["approve", "reject"]}}`. Hiding buttons in the UI is not enforcement: the resume API accepts any decision the config allows, so a crafted request could still submit `edit`. The policy lives in `interrupt_on`, not in the frontend.

**3. A 20 EUR refund executes without pausing, but a 120 EUR refund pauses for review. Which `interrupt_on` feature produces this, and what constraint must its function satisfy?**
The `when` predicate: a function of the `ToolCallRequest` returning `True` to interrupt. It must be pure, a function of the tool call alone, because it re-runs on resume: if it returns a different answer the second time, the tool may execute while the reviewer's verdict is silently discarded.

**4. You set `interrupt_before=["tools"]` on `create_agent`, the run stops, but `"__interrupt__" in result` is `False`. Is the pause real, and how do you confirm and resume it?**
Yes, it is real: static breakpoints do not produce an `__interrupt__` key. Confirm with `graph.get_state(config).next`, which reads `("tools",)` while paused before the tools node. Resume with a plain `agent.invoke(None, config)` on the same thread; no `Command(resume=...)` is needed because there is no verdict to deliver.

**5. Your sweeper finds an approval that has waited 40 minutes with no reviewer. It resumes the thread. Which decision should it submit, and what must it tell the user?**
`reject`, with a message stating the reason, for example that no reviewer was available within the SLA and the action was cancelled. Auto-approving on timeout executes an action nobody reviewed, which defeats the gate. The user must be told the action was cancelled because review timed out, and what happens next (retry, narrower action, or escalation), not left watching a spinner.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.